# Setup

In [ ]:
# ── Standard Library ─────────────────────────────────────────────────────────
import json
import re
import string
import time
import subprocess
from collections import Counter
from difflib import SequenceMatcher
import os

# ── Data Handling ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import scipy.sparse as sp

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── NLP ──────────────────────────────────────────────────────────────────────
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sentence_transformers import SentenceTransformer

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity

# ── Utilities ────────────────────────────────────────────────────────────────
from tqdm import tqdm
import requests
import gradio as gr

In [ ]:
# ── Package Installation ──────────────────────────────────────────────────────
!pip install sentence-transformers -q
!pip install gradio -q

# ── Model Downloads ───────────────────────────────────────────────────────────
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)
nltk.download("stopwords")
nltk.download("punkt")

# ── Model Loading ─────────────────────────────────────────────────────────────
nlp             = spacy.load("en_core_web_sm")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
stop_words      = set(stopwords.words("english"))

# WEEK 1 COURSEWORK

# Step 1 : Load and Explore the Data

In [ ]:
# ── Load ArXiv Dataset ──────────────────────────────────────────

# Store JSON records temporarily
# List to hold parsed JSON objects before converting to a DataFrame.
records = []

# Open ArXiv metadata file
# Safely opens the file; automatically closes it when finished.
with open('arxiv-metadata-oai-snapshot.json', 'r') as f:

    # Read first 50,000 papers
    # Loops line-by-line to avoid loading the whole file into RAM at once.
    for i, line in enumerate(f):

        # Stops the loop once 50,000 rows are collected.
        if i >= 50000:
            break

        # Convert JSON line into dictionary
        # Parses the raw text string into a Python dict and adds it to the list.
        records.append(json.loads(line))

# Create raw dataframe
# Converts the collected list of dictionaries into a structured tabular DataFrame.
df_raw = pd.DataFrame(records)

# Display dataset shape and columns
# Outputs the grid dimensions (rows, columns) of the loaded data.
print(f'Shape: {df_raw.shape}')
# Outputs the names of all columns in the DataFrame.
print(f'Columns: {df_raw.columns}')

# Preview first 3 rows
# Shows the first 3 rows of the DataFrame for a quick sanity check.
df_raw.head(3)

In [ ]:
# How many papers are in the dataset?
# Calculates total rows using len() and formats with commas for readability (e.g., 50,000).
print(f"Number of papers: {len(df_raw):,}")

# What columns/fields are available?
print(f"\nColumns available:")
# Converts the pandas Index of column names into a standard Python list and prints it.
print(df_raw.columns.tolist())

In [ ]:
# One sample record
print("Sample record:")
# Isolates specific columns, grabs the first row, and prints it without truncation.
print(df_raw[["id", "title", "abstract", "categories"]].head(1).to_string())


# Title length stats
# Splits the title text by spaces and counts the number of resulting words per row.
df_raw["title_length"] = df_raw["title"].str.split().str.len()
print(f"\nTitle length (words):")
# Generates summary statistics (mean, min, max, quartiles) for the title word counts.
print(df_raw["title_length"].describe())

# How many unique authors?
# Counts the number of completely unique strings in the raw 'authors' column.
print(f"\nUnique authors: {df_raw['authors'].nunique():,}")

# Earliest and latest submissions (version history)
print(f"\nSample of submitters:")
# Counts occurrences of each submitter name and outputs the top 10 most frequent ones.
print(df_raw["submitter"].value_counts().head(10))

In [ ]:
# ── Abstract Length Analysis ────────────────────────────────────

# Count how many words appear in each abstract
# Converts abstracts to strings, splits them by whitespace, and counts the tokens per row.
df_raw["abstract_word_count"] = (
    df_raw["abstract"]
    .astype(str)
    .str.split()
    .str.len()
)

# Show summary statistics for abstract lengths
print("\nAbstract length statistics:")
# Outputs count, mean, standard deviation, min, max, and percentiles of abstract word counts.
print(df_raw["abstract_word_count"].describe())

# Plot distribution of abstract lengths
# Initializes a plotting canvas with a width of 10 inches and height of 6 inches.
plt.figure(figsize=(10, 6))

# Generates a histogram grouped into 50 bars to visualize word count frequencies.
plt.hist(
    df_raw["abstract_word_count"],
    bins=50
)

# Adds descriptive labels and title to the chart axes.
plt.title("Distribution of Abstract Lengths")
plt.xlabel("Number of Words in Abstract")
plt.ylabel("Number of Papers")

# Adjusts subplot padding to prevent text/labels from getting cut off.
plt.tight_layout()
# Renders the finalized plot on the screen.
plt.show()

In [ ]:
# ── Missing Values Analysis ─────────────────────────────────────

# Check how many missing values exist in each column
print(f"\nMissing values per column:")

# Detects null/NaN entries per cell and sums them up column by column.
print(df_raw.isnull().sum())

# Store missing value counts for visualisation
# Saves the resulting series of null counts into a variable for plotting.
missing_values = df_raw.isnull().sum()

# Plot missing values across columns
# Sets up a 10x6 inch blank canvas for the visualization.
plt.figure(figsize=(10, 6))

# Generates a vertical bar chart directly from the pandas Series data.
missing_values.plot(kind="bar")

# Adds descriptive context titles and axis labels to the chart.
plt.title("Missing Values per Column")
plt.xlabel("Columns")
plt.ylabel("Number of Missing Values")

# Rotates the x-axis column labels by 45 degrees so they don't overlap.
plt.xticks(rotation=45)

# Automatically fits the plot elements cleanly within the figure boundaries.
plt.tight_layout()
# Displays the completed bar chart.
plt.show()

In [ ]:
# Check how many unique category combinations exist
print(
    f"\nSample unique categories: "
    f"{df_raw['categories'].nunique()}"
)

In [ ]:
# ── Category Analysis ───────────────────────────────────────────

# Split category strings into individual categories
# Splits space-separated category strings into lists, then flattens (explodes) those lists into individual rows.
all_categories = df_raw["categories"].str.split().explode()

# Count unique individual categories
# Calculates the total number of distinct, individual categories found across all papers.
print(f"\nNumber of unique categories: {all_categories.nunique()}")

# Display all unique categories
print("\nAll unique categories:")
# Extracts all unique category labels and sorts them alphabetically.
print(sorted(all_categories.unique()))

# Find the most common categories
# Groups the flattened categories and tallies their individual frequencies.
category_counts = all_categories.value_counts()

print("\nTop 20 most common categories:")
# Extracts the 20 highest-frequency categories from the value counts.
print(category_counts.head(20))

# Count how many categories each paper has
# Splits the categories string by whitespace and records the number of labels assigned to each paper.
df_raw["num_categories"] = (
    df_raw["categories"]
    .str.split()
    .str.len()
)

# Compare single-label vs multi-label papers
# Filters and sums rows where the category count is exactly equal to 1.
print(
    f"\nPapers with single category: "
    f"{(df_raw['num_categories'] == 1).sum():,}"
)

# Filters and sums rows where the category count is greater than 1 (multi-label classification setup).
print(
    f"Papers with multiple categories: "
    f"{(df_raw['num_categories'] > 1).sum():,}"
)

# Plot the top 10 most common categories
# Isolates the top 10 categories for visualization.
top_categories = category_counts.head(10)

# Sets up a 10x6 inch plot window.
plt.figure(figsize=(10, 6))

# Renders the top 10 frequencies as a vertical bar chart.
top_categories.plot(kind="bar")

# Customizes the chart with appropriate titles and axis descriptions.
plt.title("Top 10 Categories in the ArXiv Dataset")
plt.xlabel("Category")
plt.ylabel("Number of Papers")

# Rotates category name labels along the x-axis to ensure legibility.
plt.xticks(rotation=45)

# Tightens layout margins to clean up whitespace around the figure.
plt.tight_layout()
# Draws the final bar chart to the screen.
plt.show()

# Step 2: Handle Categories (Labels)

In [ ]:
# ── Step 2: Handle Categories ──────────────────────────────────

# Create a working copy so df_raw stays unchanged for EDA/reference
# Creates a deep copy of the original DataFrame to prevent unexpected side effects or warnings.
df_work = df_raw.copy()

# ArXiv papers can have multiple categories, so use the first one as the primary label
# Splits the category string by whitespace and grabs the first element index [0] as the primary category.
df_work["main_category"] = df_work["categories"].str.split().str[0]

# Extract the broad subject area, e.g. "cs.LG" becomes "cs"
# Splits the main category string at the period separator and keeps the left-side prefix.
df_work["subject_area"] = df_work["main_category"].str.split(".").str[0]

# Find older standalone categories that do not use dot notation
# Flattens all categories across the dataset and isolates the unique set of individual tags.
all_cats = df_work["categories"].str.split().explode().unique()

# Uses a list comprehension to identify raw category tags that lack a sub-discipline period marker.
no_dot = [
    c for c in all_cats
    if "." not in c
]

print(f"Standalone categories found (no dot): {no_dot}")
# Checks how many papers have a main category that matches this legacy, non-dotted format.
print(f"Papers affected: {df_work['main_category'].isin(no_dot).sum():,}")

# Map older standalone ArXiv categories into broader subject areas
# Dictionary defining the translation layout to group specialized legacy codes into unified scientific domains.
standalone_map = {
    # High Energy Physics
    "hep-th"  : "physics",
    "hep-ph"  : "physics",
    "hep-ex"  : "physics",
    "hep-lat" : "physics",

    # Relativity and Quantum Physics
    "gr-qc"   : "physics",
    "quant-ph": "physics",

    # Nuclear Physics
    "nucl-th" : "physics",
    "nucl-ex" : "physics",

    # Mathematical Physics and Astrophysics
    "math-ph" : "physics",
    "astro-ph": "physics",

    # Condensed Matter and Nonlinear Sciences
    "cond-mat" : "physics",
    "nlin"     : "physics",

    # Existing broad categories
    "q-bio" : "q-bio",
    "q-fin" : "q-fin",
    "cs"    : "cs",
    "math"  : "math",
    "stat"  : "stat"
}

# Apply manual mapping so all labels follow the same broad-category format
# Replaces specific legacy strings with their mapped macro-domain names using the dictionary.
df_work["subject_area"] = df_work["subject_area"].replace(
    standalone_map
)

print(f"\nUnique main categories (fine-grained) : {df_work['main_category'].nunique()}")
print(f"Unique subject areas   (broad)        : {df_work['subject_area'].nunique()}")

print(f"\nBroad subject area distribution:")
# Displays the final record counts grouped by the newly standardized macro subject areas.
print(df_work["subject_area"].value_counts())

In [ ]:
# ── Broad Subject Area Distribution ────────────────────────────

# Plot how papers are distributed across the broad subject areas
# Chains value_counts() to get category frequencies and immediately generates a bar chart wrapper.
df_work["subject_area"].value_counts().plot(
    kind="bar",
    figsize=(10,6)
)

# Adds the descriptive main header and labels for the axes.
plt.title("Distribution of Broad Subject Areas")
plt.xlabel("Subject Area")
plt.ylabel("Number of Papers")

# Angles the subject names by 45 degrees to prevent overlapping text on the x-axis.
plt.xticks(rotation=45)

# Trims excess padding around the plot elements to ensure a clean layout.
plt.tight_layout()
# Flushes the figure buffer and displays the completed plot window.
plt.show()

In [ ]:
# View one full record after category handling
# This helps check whether main_category and subject_area were created correctly
print(df_work.iloc[472].to_string())

In [ ]:
# ── Check Less Common Subject Areas ─────────────────────────────

# Check whether Electrical Engineering & Systems Science appears in the dataset
# Filters the working DataFrame using a boolean mask to find rows containing the substring "eess".
ee_papers = df_work[
    df_work["categories"].str.contains(
        "eess",
        na=False # Prevents errors by treating missing/NaN category values as False.
    )
]

print(f"Electrical Engineering & Systems Science papers: {len(ee_papers):,}")

print(f"\nCategories found:")

print(
    # Isolates, splits, and flattens the category tags specifically for EESS-matched papers to count sub-groups.
    ee_papers["categories"]
    .str.split()
    .explode()
    .value_counts()
)

# Check whether Economics-related papers appear in the dataset
# Filters the working DataFrame using a boolean mask to find rows containing the substring "econ".
econ_papers = df_work[
    df_work["categories"].str.contains(
        "econ",
        na=False # Ignores missing/NaN category fields during evaluation.
    )
]

print(f"Economics papers: {len(econ_papers):,}")

print(f"\nCategories found:")

print(
    # Isolates, splits, and flattens the category tags specifically for Economics-matched papers to count sub-groups.
    econ_papers["categories"]
    .str.split()
    .explode()
    .value_counts()
)

# Step 3: Text Preprocessing

In [ ]:
# ── Preprocessing Setup ─────────────────────────────────────────

# Load spaCy model for tokenisation, POS tagging, NER and lemmatisation
# Loads the small English pipeline engine optimized for fast, rule-based text breakdown and linguistic analysis.
nlp = spacy.load("en_core_web_sm")

# Load English stopwords for filtering common low-information words
# Imports NLTK's standard stopword list and casts it to a set for O(1) hash-table lookup speeds during text filtering.
STOPWORDS = set(stopwords.words("english"))

print("Libraries loaded successfully")
# Prints the total size of the imported baseline stopword vocabulary.
print(f"Number of stopwords: {len(STOPWORDS)}")

In [ ]:
# ── 3a: Text Cleaning ───────────────────────────────────────────

def clean_text(text):
    """
    Cleans raw text by:
    - Converting to lowercase
    - Removing LaTeX/math expressions common in scientific abstracts
    - Removing URLs
    - Replacing hyphens with spaces
    - Removing punctuation and special characters
    - Removing extra whitespace
    """
    # Type-guard check: returns an empty string if the entry is null, numeric, or non-string.
    if not isinstance(text, str):
        return ""

    # Convert all text to lowercase for consistency
    # Normalizes tokens (e.g., 'The' and 'the' become identical) to reduce vocabulary variance.
    text = text.lower()

    # Remove simple LaTeX expressions and commands
    # Replaces inline math enclosed in dollar signs (e.g., $x^2$) with an empty string.
    text = re.sub(r'\$.*?\$', '', text)
    # Replaces common LaTeX command structures like \textbf{word} with an empty string.
    text = re.sub(r'\\[a-zA-Z]+\{.*?\}', '', text)

    # Remove web links if any appear in the text
    # Truncates any HTTP/HTTPS URLs or standard domain prefixes found in the body.
    text = re.sub(r'http\S+|www\S+', '', text)

    # Replace hyphens with spaces so joined terms remain readable
    # Splits composite words (e.g., 'state-of-the-art' into 'state of the art') instead of smashing them together.
    text = re.sub(r'-', ' ', text)

    # Keep only letters, numbers and spaces
    # Strips out punctuation, brackets, symbols, and special unicode characters.
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # Remove extra spaces
    # Collapses consecutive spaces, tabs, or newlines into a single space, then trims the outer edges.
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply cleaning to titles and abstracts
# Vectorizes the processing function across every row of the specified text columns using .apply().
df_work["clean_title"] = df_work["title"].apply(clean_text)
df_work["clean_abstract"] = df_work["abstract"].apply(clean_text)

# Compare original and cleaned abstract
print("Original abstract:")
# Extracts the very first raw item using integer-location based indexing.
print(df_work["abstract"].iloc[0])

print("\nCleaned abstract:")
# Extracts the corresponding processed item to verify the regex transformations visually.
print(df_work["clean_abstract"].iloc[0])

# Compare original and cleaned title
print("Original title:")
print(df_work["title"].iloc[0])

print("\nCleaned title:")
print(df_work["clean_title"].iloc[0])

In [ ]:
# ── 3b: Tokenisation, Stopword Removal and Lemmatisation ────────

def preprocess_spacy(text):
    """
    Processes cleaned text by:
    - Tokenising text into individual words
    - Removing stopwords and non-alphabetic tokens
    - Lemmatising words into their base form

    Returns token lists and final lemma text for modelling.
    """

    # Passes raw text through the spaCy pipeline, creating a Doc object populated with structural annotations.
    doc = nlp(text)

    # Keep alphabetic word tokens
    # Extracts raw string text from tokens, filtering out numbers, punctuation, or symbols via .is_alpha.
    tokens = [
        token.text for token in doc
        if token.is_alpha
    ]

    # Remove stopwords and keep meaningful alphabetic words
    # Filters out common syntactic noise (e.g., 'and', 'the') using spaCy's built-in stopword dictionary.
    filtered_tokens = [
        token.text for token in doc
        if not token.is_stop and token.is_alpha
    ]

    # Convert filtered tokens into base word forms
    # Resolves words to their structural root forms (e.g., 'running' -> 'run', 'studies' -> 'study') using .lemma_.
    lemmas = [
        token.lemma_ for token in doc
        if not token.is_stop and token.is_alpha
    ]

    # packages components into a dictionary to return multiple granular representations of the text.
    return {
        "tokens": tokens,
        "filtered_tokens": filtered_tokens,
        "lemmas": lemmas,
        "lemma_text": " ".join(lemmas) # Concatenates the root word array back into a single whitespace-separated string.
    }


print("Preprocessing function defined.")
print("Testing on sample abstract...")

# Test on one abstract and one title before applying to the full dataset
# Passes the first row of cleaned abstract text into the processor to verify performance.
sample_abstract = preprocess_spacy(df_work["clean_abstract"].iloc[0])

# Passes the first row of cleaned title text into the processor for validation.
sample_title = preprocess_spacy(df_work["clean_title"].iloc[0])

# Displays sliced subsets of the resulting lists and strings to inspect the workflow transformations.
print(f"\nTokens (first 10): {sample_abstract['tokens'][:10]}")
print(f"After stopwords (first 25): {sample_abstract['filtered_tokens'][:25]}")
print(f"Lemmas (first 50): {sample_abstract['lemmas'][:50]}")

print(f"\nFinal abstract lemma text:\n{sample_abstract['lemma_text'][:200]}")

print(f"\nTitle tokens (first 10): {sample_title['tokens'][:10]}")
print(f"Title after stopwords (first 25): {sample_title['filtered_tokens'][:25]}")
print(f"Title lemmas (first 50): {sample_title['lemmas'][:50]}")

print(f"\nFinal title lemma text:\n{sample_title['lemma_text'][:200]}")

In [ ]:
# ── Apply preprocessing to the full dataset ────────────────────

# Process abstracts and titles in batches using spaCy
# nlp.pipe() is much faster than looping row-by-row

print("Processing abstracts and titles...")

# Streams texts into the spaCy pipeline as a generator and aggregates them into a list of Doc objects.
abstract_docs = list(
    nlp.pipe(
        df_work["clean_abstract"],
        batch_size=500 # Groups 500 records per internal buffer to maximize multi-threading/vectorized execution speeds.
    )
)

# Streams and processes title entries in efficient micro-batches.
title_docs = list(
    nlp.pipe(
        df_work["clean_title"],
        batch_size=500 # Passes documents through the tokenization/tagging pipeline in 500-item chunks.
    )
)

# Create final lemmatised abstract text for modelling
# List comprehension that loops over the parsed Doc objects, filters tokens, and joins lemmas into clean strings.
df_work["lemma_abstract"] = [
    " ".join([
        token.lemma_
        for token in doc
        if not token.is_stop and token.is_alpha # Filters out structural stop words and keeps only purely alphabetic words.
    ])
    for doc in abstract_docs
]

# Create final lemmatised title text for modelling
# Rebuilds normalized title strings by extracting root lemma forms from the parsed title documents.
df_work["lemma_title"] = [
    " ".join([
        token.lemma_
        for token in doc
        if not token.is_stop and token.is_alpha # Strips functional noise words and non-alphabetic elements.
    ])
    for doc in title_docs
]

print(f"Done! Processed {len(df_work):,} rows")

# Preview processed abstract text
print("\nSample abstract lemma text:")
# Prints the first 300 characters of the processed abstract string from the first row.
print(df_work["lemma_abstract"].iloc[0][:300])

# Preview processed title text
print("\nSample title lemma text:")
# Prints the fully processed title string from the first row.
print(df_work["lemma_title"].iloc[0])

In [ ]:
# ── Quick Check: Before vs After Preprocessing ──────────────────

# Change this index to inspect different papers
# Centralized variable to alter which specific paper row is pulled across all printing blocks.
idx = 5

# Compare original, cleaned and lemmatised title
print("=== ORIGINAL TITLE ===")
# Fetches the raw title as it was loaded from the original source dataset.
print(df_work["title"].iloc[idx])

print("\n=== CLEAN TITLE ===")
# Fetches the title after lowercasing, stripping LaTeX, and removing punctuation.
print(df_work["clean_title"].iloc[idx])

print("\n=== LEMMATISED TITLE USED FOR ML ===")
# Fetches the final tokenized, stopword-filtered, and root-normalized string ready for machine learning features.
print(df_work["lemma_title"].iloc[idx])

# Compare original, cleaned and lemmatised abstract
print("\n=== ORIGINAL ABSTRACT ===")
# Fetches the raw block of abstract text from the source dataset.
print(df_work["abstract"].iloc[idx])

print("\n=== CLEAN ABSTRACT ===")
# Fetches the abstract with extra spaces, punctuation, links, and math syntax stripped.
print(df_work["clean_abstract"].iloc[idx])

print("\n=== LEMMATISED ABSTRACT USED FOR ML ===")
# Fetches the processed string showing only the lemmatized keywords that carry semantic weight.
print(df_work["lemma_abstract"].iloc[idx])

## Label Encoding

In [ ]:
# ── Step 5: Label Encoding ─────────────────────────────────────

# Convert subject labels into numeric values for machine learning
# Instantiates Scikit-learn's preprocessing utility to map categorical strings to unique integers.
le = LabelEncoder()

# Fits the encoder to the unique subject areas and transforms the text column into integer labels.
df_work["category_encoded"] = le.fit_transform(
    df_work["subject_area"]
)

# Print the mapping so predictions can be decoded later
print("Label Encoding Mapping:")

# Iterates through the sorted list of unique classes stored inside the fitted encoder object.
for i, label in enumerate(le.classes_):
    # Displays the clean text string and its corresponding machine-readable integer index.
    print(f"  {label} → {i}")

# Outputs the grand total number of distinct target classes found in the dataset.
print(f"\nTotal categories: {len(le.classes_)}")

# Quick check that labels were encoded correctly
print(f"\nSample check:")

print(
    # Pulls the first 100 rows to manually verify that the text matches the assigned integer code.
    df_work[
        [
            "subject_area",
            "category_encoded"
        ]
    ].head(100)
)

## Export CSV

In [ ]:
# ── Step 5: Export Final Processed Dataset ─────────────────────

# Final modelling dataframe used throughout the ML pipeline
# Includes both original and processed text columns
# Subsets the working DataFrame to isolate only the core columns needed for training and evaluation.
# Uses .copy() to decouple this final dataset from df_work, allocating a fresh block of memory.
df_model = df_work[
    [
        "id",
        "title",
        "abstract",
        "clean_title",
        "clean_abstract",
        "lemma_title",
        "lemma_abstract",
        "subject_area",
        "category_encoded"
    ]
].copy()

# Export full processed dataset for coursework submission
# Writes the cleaned dataset to disk as a CSV file.
# index=False prevents pandas from writing the numeric row index as an extra, unnamed column.
df_model.to_csv(
    "arxiv_processed.csv",
    index=False
)

# Save label mapping separately for decoding predictions later
# Reconstructs the target class indices into a clean dictionary mapping for easy JSON serialization.
mapping = {
    label: int(i) # Explicitly casts numpy integers to native Python ints to prevent JSON encoding errors.
    for i, label in enumerate(le.classes_)
}

# Context manager opens a new JSON file in write mode ('w').
with open("label_mapping.json", "w") as f:
    # Serializes the dictionary into structured text with a clean 2-space indentation format.
    json.dump(mapping, f, indent=2)

print(
    f"CSV saved: "
    f"{len(df_model):,} rows, "
    f"{len(df_model.columns)} columns"
)

print(f"Columns: {df_model.columns.tolist()}")

print(f"\nLabel mapping saved to label_mapping.json")

# Preview exported dataset
print(f"\nFirst 3 rows of exported CSV:")

# Prints a structural snapshot of the top 3 rows inside the new df_model DataFrame.
print(df_model.head(3))

# Display exported file size
# Fetches the raw file footprint in bytes, then divides by 1024 twice to convert it to Megabytes.
file_size = os.path.getsize(
    "arxiv_processed.csv"
) / (1024 * 1024)

# Outputs the calculated file footprint formatted neatly to 2 decimal places.
print(f"File size: {file_size:.2f} MB")

In [ ]:
# Reload exported CSV to verify it was saved correctly
df_check = pd.read_csv("arxiv_processed.csv")
df_check.head(10)
print(df_model.columns)

In [ ]:
# ── Remove Duplicate Processed Abstracts ─────────────────────

# Prints the baseline shape of the dataframe before scanning for duplicate content.
print(f"Before removing duplicates: {df_model.shape}")

# Drops rows where the 'lemma_abstract' text is identical, leaving only the first occurrence.
# .reset_index(drop=True) discards the old fragmented index and builds a clean 0-indexed sequence.
df_model = df_model.drop_duplicates(
    subset="lemma_abstract"
).reset_index(drop=True)

# Prints the new, reduced shape to confirm how many duplicate rows were removed.
print(f"After removing duplicates: {df_model.shape}")

# WEEK 2 COURSEWORK

In [ ]:
# ── Train/Test Split ───────────────────────────────────────

# Use lemmatised title and abstract as text inputs
# Extracts the independent feature columns containing the cleaned text tokens for modeling.
X_abstract = df_model["lemma_abstract"]
X_title = df_model["lemma_title"]

# Use encoded category as the target label
# Assigns the dependent variable (target integer codes) to y.
y = df_model["category_encoded"]

# Split before fitting vectorisers to prevent train-test leakage
# Stratification keeps class distribution similar in train and test sets
# Splitting arrays simultaneously ensures indices match exactly across both features (abstracts and titles).
(
    X_abs_train,
    X_abs_test,
    X_ttl_train,
    X_ttl_test,
    y_train,
    y_test
) = train_test_split(
    X_abstract,
    X_title,
    y,
    test_size=0.2,       # Allocates 20% of the dataset for testing and evaluation, leaving 80% for training.
    random_state=42,     # Sets a static random seed to guarantee reproducibility across multiple runs.
    stratify=y           # Forces the training and test splits to preserve the original percentage of each target class.
)

# Prints the precise breakdown count of samples assigned to the training versus evaluation groups.
print(f"Train size: {len(y_train):,} | Test size: {len(y_test):,}")

# Step 1: Feature Representation

## TF-IDF Vectorisation

In [ ]:
# ── Input Configuration 1 – Abstract Unigrams ──────────────
# Baseline TF-IDF representation using only single words from abstracts
# This tests how much category signal is available from individual terms

# Initializes a TF-IDF vectorizer restricted to individual words (unigrams).
tfidf_abs_uni = TfidfVectorizer(
    max_features=50_000,   # Keeps only the top 50,000 most frequent tokens across the corpus to limit dimensionality.
    ngram_range=(1, 1),    # Sets token boundaries to strictly capture single words.
    sublinear_tf=True,     # Applies logarithmic scaling (1 + log(tf)) to dampen the influence of highly repetitive words.
    min_df=3               # Drops rare words that appear in fewer than 3 distinct documents to filter out noise.
)

# Learns the vocabulary dictionary from the training data and returns a sparse document-term matrix.
X_train_c1 = tfidf_abs_uni.fit_transform(X_abs_train)
# Transforms the test data into a sparse feature matrix using the exact vocabulary learned from training.
X_test_c1 = tfidf_abs_uni.transform(X_abs_test)

print(f"Config 1 – Abstract unigrams: {X_train_c1.shape}")


# ── 1d: Input Configuration 2 – Abstract Unigrams + Bigrams ────
# Adds two-word phrases to capture scientific terms such as "neural network"
# Comparing with Config 1 shows whether bigrams improve representation

# Initializes a vectorizer that evaluates both single tokens and consecutive word pairings.
tfidf_abs_bi = TfidfVectorizer(
    max_features=50_000,   # Re-caps total features to 50,000, prioritizing the most important unigrams and bigrams combined.
    ngram_range=(1, 2),    # Extends boundaries to capture both single words and two-word sequences (e.g., 'machine learning').
    sublinear_tf=True,     # Uses sublinear term frequency scaling to compress extreme word counts.
    min_df=3               # Ignores tokens or word pairs that appear less than 3 times across the entire split.
)

# Fits the model vocabulary and extracts a mixed unigram/bigram sparse training matrix.
X_train_c2 = tfidf_abs_bi.fit_transform(X_abs_train)
# Vectorizes the validation/test partition based on the trained model configuration.
X_test_c2 = tfidf_abs_bi.transform(X_abs_test)

print(f"Config 2 – Abstract unigrams + bigrams: {X_train_c2.shape}")


# ── 1e: Input Configuration 3 – Title + Abstract ───────────────
# Combines title and abstract features for a richer representation
# Titles are shorter, so a smaller TF-IDF vocabulary is used for them

# Creates a distinct vectorizer pipeline tuned explicitly for the shorter, more dense language of titles.
tfidf_ttl = TfidfVectorizer(
    max_features=20_000,   # Restricts vocabulary to a tighter 20,000 keyword limit suited for short strings.
    ngram_range=(1, 2),    # Evaluates both single title keywords and two-word combinations.
    sublinear_tf=True,     # Scales term frequencies logarithmically.
    min_df=2               # Sets a lower threshold to capture terms occurring at least twice across the dataset.
)

# Establishes the distinct vocabulary for title components based on the training sub-array.
X_train_ttl = tfidf_ttl.fit_transform(X_ttl_train)
# Maps test titles into the established title vocabulary matrix space.
X_test_ttl = tfidf_ttl.transform(X_ttl_test)

# Uses SciPy's sparse column-wise stack tool to stitch the title matrix to the side of the Configuration 2 abstract matrix.
X_train_c3 = sp.hstack([X_train_ttl, X_train_c2])
# Concatenates the title and abstract features for the test split to ensure identical matrix width.
X_test_c3 = sp.hstack([X_test_ttl, X_test_c2])

print(f"Config 3 – Title + abstract unigrams + bigrams: {X_train_c3.shape}")


# ── 1f: Feature Matrix Summary ─────────────────────────────────
# Summarise how many features each representation produced

print("\nFeature matrix summary:")

# Extracts the width component (index 1) of the matrix shapes to show the exact column count.
print(f"  Config 1 (abstract unigrams)        : {X_train_c1.shape[1]:,} features")
print(f"  Config 2 (abstract uni+bigrams)     : {X_train_c2.shape[1]:,} features")
print(f"  Config 3 (title + abstract features): {X_train_c3.shape[1]:,} features")

print(f"\nClass distribution in train set:")

print(
    # Converts target integer codes to a Series, maps them back to text categories using a dictionary, and returns value counts.
    pd.Series(y_train)
    .map(dict(enumerate(le.classes_)))
    .value_counts()
)

## Doc2Vec

In [ ]:
# ──  Advanced Feature Representation – Doc2Vec ──────────────
# Doc2Vec creates one dense vector for each paper.
# This gives a semantic-style representation to compare against TF-IDF.

# Combine title and abstract so each paper is represented as one document
# Concatenates the string series column-wise with a space separator, then converts the result into a native Python list.
X_train_text = (X_ttl_train + " " + X_abs_train).tolist()
X_test_text = (X_ttl_test + " " + X_abs_test).tolist()

# Tokenise each document using simple whitespace splitting
# Converts each raw text string into a list of constituent word tokens.
train_tokens = [
    doc.split()
    for doc in X_train_text
]

test_tokens = [
    doc.split()
    for doc in X_test_text
]

# Create tagged documents required by Doc2Vec
# Wraps token lists in Gensim's TaggedDocument format, mapping each document array to a unique string ID.
tagged_train_docs = [
    TaggedDocument(
        words=tokens,
        tags=[str(i)]
    )
    for i, tokens in enumerate(train_tokens)
]

# Train Doc2Vec model on training documents only
# Initializes the model structure with chosen vector dimensions, structural architectures, and execution boundaries.
doc2vec_model = Doc2Vec(
    vector_size=100,  # Specifies the target length (dimensionality) of the final dense document embeddings.
    window=5,         # Sets the maximum context distance between the current and predicted word within a sentence.
    min_count=3,      # Discards words with a total frequency lower than 3 across the training split.
    workers=4,        # Allocates 4 parallel CPU threads for faster background processing and model optimization.
    epochs=20,        # Sets the number of passes (iterations) the model makes over the entire dataset during training.
    dm=1,             # Activates Distributed Memory (PV-DM) architecture, which preserves word order context.
    seed=42           # Locks the random number generator seed to ensure reproducible neural network initial weights.
)

# Constructs the model's internal vocabulary indexing by scanning the unique words in the tagged training set.
doc2vec_model.build_vocab(tagged_train_docs)

# Optimizes the neural network weights by training over the corpus for the designated number of epochs.
doc2vec_model.train(
    tagged_train_docs,
    total_examples=doc2vec_model.corpus_count,
    epochs=doc2vec_model.epochs
)

# Extract dense vectors for training documents
# Retrieves the learned document vectors directly from the trained model's docvecs (.dv) lookup table.
X_train_c4 = np.array([
    doc2vec_model.dv[str(i)]
    for i in range(len(tagged_train_docs))
])

# Infer dense vectors for unseen test documents
# Computes embeddings for the evaluation set by freezing the word weights and performing gradient descent on the new tokens.
X_test_c4 = np.array([
    doc2vec_model.infer_vector(tokens)
    for tokens in test_tokens
])

print(f"Config 4 – Doc2Vec title + abstract embeddings: {X_train_c4.shape}")
print(f"Config 4 – Doc2Vec test embeddings:           {X_test_c4.shape}")

# Step 2 : Text Classification

## SVM vs LogisticRegression vs MultinomiaNB

In [ ]:
# ── Full Classification Experiment ─────────────────────
# Compare different feature representations across multiple ML models.
# This helps identify which model + representation works best.

# ── Feature Configurations ─────────────────────────────────────
# Nested dictionary mapping human-readable experiment names to their respective training/testing matrices.
feature_configs = {
    "Config 1: TF-IDF Abstract Unigrams": {
        "X_train": X_train_c1,
        "X_test": X_test_c1,
        "type": "tfidf" # flag to help condition processing loops later
    },

    "Config 2: TF-IDF Abstract Uni+Bigrams": {
        "X_train": X_train_c2,
        "X_test": X_test_c2,
        "type": "tfidf"
    },

    "Config 3: TF-IDF Title+Abstract Uni+Bigrams": {
        "X_train": X_train_c3,
        "X_test": X_test_c3,
        "type": "tfidf"
    },

    "Config 4: Doc2Vec Title+Abstract": {
        "X_train": X_train_c4,
        "X_test": X_test_c4,
        "type": "doc2vec"
    }
}

# ── Models ─────────────────────────────────────────────────────
# Linear SVM and Logistic Regression use class_weight to handle imbalance.
# Multinomial NB is included as a simple probabilistic baseline.
models = {
    "Linear SVM": LinearSVC(
        random_state=42,          # Locks the random number generator seed to ensure identical coefficients across runs.
        max_iter=5000,            # Raises optimization iterations to prevent convergence warnings on sparse datasets.
        class_weight="balanced"   # Auto-adjusts weights inversely proportional to class frequencies to combat class imbalance.
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=1000,            # Higher threshold to ensure the gradient descent algorithm hits global convergence.
        class_weight="balanced",  # penalizes minority class misclassifications more heavily.
        random_state=42,          # Enforces reproducible solver operations.
        n_jobs=-1                 # Leverages all available CPU cores to execute optimization routines in parallel.
    ),

    "Multinomial Naive Bayes": MultinomialNB() # Simple, fast baseline that assumes absolute feature independence.
}

# ── Training + Evaluation ──────────────────────────────────────

# Empty array to collect dictionary objects containing performance benchmarks for every permutation.
results = []

# Double loop matrix that fits every selected algorithm variant against every text representation pipeline.
for model_name, model in models.items():

    for config_name, config in feature_configs.items():

        # Naive Bayes is skipped for Doc2Vec because dense vectors can contain negative values
        # MultinomialNB calculates word probabilities and throws a ValueError if it encounters negative inputs.
        if model_name == "Multinomial Naive Bayes" and config["type"] == "doc2vec":
            print(f"Skipping {model_name} on {config_name} (negative dense embeddings)")
            continue

        print(f"\nTraining {model_name} on {config_name}...")

        # Captures the start epoch timestamp in seconds to monitor processing duration.
        start_time = time.time()

        # Fits the structural algorithm weights using the active loop matrix variant.
        model.fit(config["X_train"], y_train)
        # Generates label predictions using the evaluation feature matrix split.
        y_pred = model.predict(config["X_test"])

        # Calculates total elapsed time for training and inference operations.
        runtime = time.time() - start_time

        # Computes performance metrics comparing test ground truth labels to the model predictions.
        accuracy = accuracy_score(y_test, y_pred)
        # Macro F1 calculates unweighted metrics per class, highlighting performance on small minority classes.
        macro_f1 = f1_score(y_test, y_pred, average="macro")
        # Weighted F1 factors in class support size, giving a reflection of overall system correctness.
        weighted_f1 = f1_score(y_test, y_pred, average="weighted")

        # Stores performance data inside a structured tracking array.
        results.append({
            "Model": model_name,
            "Configuration": config_name,
            "Accuracy": accuracy,
            "Macro F1": macro_f1,
            "Weighted F1": weighted_f1,
            "Runtime (seconds)": runtime
        })

        print(f"Accuracy:    {accuracy:.4f}")
        print(f"Macro F1:    {macro_f1:.4f}")
        print(f"Weighted F1: {weighted_f1:.4f}")
        print(f"Runtime:     {runtime:.2f} seconds")

# ── Results Table ──────────────────────────────────────────────

# Converts the array of result dictionary records into a tabular pandas DataFrame.
ml_results_df = pd.DataFrame(results)

# Sort by Macro F1 because the dataset is imbalanced
# Places the pipeline variant with the best minority-class classification performance at index 0.
ml_results_df = ml_results_df.sort_values(
    by="Macro F1",
    ascending=False
)

print("\nFull Experimental Results:")
# Leverages IPython's rich display system to output a neatly formatted tabular representation of the metrics.
display(ml_results_df)

# ── Best Overall Pipeline ──────────────────────────────────────

# Extracts the top-performing model setup row using structural location mapping.
best_pipeline = ml_results_df.iloc[0]

print("\nBest Overall Pipeline:")
print(f"Model:         {best_pipeline['Model']}")
print(f"Configuration: {best_pipeline['Configuration']}")
print(f"Accuracy:      {best_pipeline['Accuracy']:.4f}")
print(f"Macro F1:      {best_pipeline['Macro F1']:.4f}")
print(f"Weighted F1:   {best_pipeline['Weighted F1']:.4f}")
print(f"Runtime:       {best_pipeline['Runtime (seconds)']:.2f} seconds")

In [ ]:
# ── Visualise Experimental Results ─────────────────────────────

import seaborn as sns

# Set up the plotting style for a clean, academic look
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 8))

# Melt the DataFrame to transform metrics into a long format suitable for seaborn grouping
melted_results = ml_results_df.melt(
    id_vars=["Model", "Configuration"],
    value_vars=["Accuracy", "Macro F1"],
    var_name="Metric",
    value_name="Score"
)

# Create a combined identifier label for the y-axis to isolate every specific permutation
melted_results["Pipeline"] = (
    melted_results["Model"] + " \n(" + melted_results["Configuration"] + ")"
)

# Sort the visual representations to match the sorted order of your results DataFrame
pipeline_order = (
    ml_results_df["Model"] + " \n(" + ml_results_df["Configuration"] + ")"
).tolist()

# Plot horizontal bars grouped by the evaluation metrics
ax = sns.barplot(
    data=melted_results,
    y="Pipeline",
    x="Score",
    hue="Metric",
    order=pipeline_order,
    palette="muted"
)

# Formatting and layout adjustments
plt.title("Model Performance Comparison across Feature Configurations", fontsize=14, pad=15)
plt.xlabel("Score (0.0 - 1.0)", fontsize=12)
plt.ylabel("Pipeline Variant", fontsize=12)
plt.xlim(0, 1.0)
plt.legend(title="Evaluation Metrics", loc="lower right")

# Annotate the exact values on the ends of the bars for quick analytical scanning
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=5, fontsize=9)

plt.tight_layout()
plt.show()

# Step 3: Summarisation

In [ ]:
# ── Extractive Summarisation Function ───────────────────

def extractive_summary(text, num_sentences=2):
    """
    Generates a simple extractive summary from an abstract.

    Method:
    - Split the abstract into sentences
    - Count frequent informative words
    - Score each sentence based on word importance
    - Select the highest-scoring sentences
    """

    # Return empty output if the input is missing or invalid
    # Pre-checks validity of data to avoid runtime attribute failures with non-string structures.
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""

    # Split abstract into sentences
    # Leverages NLTK's pre-trained Punkt sentence tokenizer to reliably isolate sentence boundaries.
    sentences = sent_tokenize(text)

    # If the abstract is already short, keep it unchanged
    # Short-circuits the scoring step if the block contains fewer or equal sentences than the targeted length.
    if len(sentences) <= num_sentences:
        return text

    # Tokenise words and remove low-information tokens
    # Breaks down the global body string into lowercased token units for uniform frequency evaluations.
    words = word_tokenize(text.lower())

    # Uses a list comprehension to strip background noise words, isolated punctuation symbols, and formatting markup.
    filtered_words = [
        word for word in words
        if word not in stop_words
        and word not in string.punctuation
        and word.isalnum() # Filters for alphanumeric sequences to avoid rogue characters.
    ]

    # Count important word frequencies
    # Instantiates a high-performance hash collection to map keywords directly to their overall count values.
    word_freq = Counter(filtered_words)

    sentence_scores = {}

    # Score each sentence using the frequency of important words
    # Outer loop parses through individual sentence structures in sequence.
    for sentence in sentences:

        # Segments the localized single sentence back down into lowercase sub-words.
        sentence_words = word_tokenize(sentence.lower())

        for word in sentence_words:

            # Checks if the token exists inside our global keyword importance hash dictionary.
            if word in word_freq:
                # Dynamically updates cumulative points for the specific sentence entry.
                sentence_scores[sentence] = (
                    sentence_scores.get(sentence, 0)
                    + word_freq[word]
                )

        # Normalise score so longer sentences are not unfairly favoured
        # Divides raw cumulative importance points by sentence word count to establish density balance.
        if len(sentence_words) > 0:
            sentence_scores[sentence] = (
                sentence_scores.get(sentence, 0)
                / len(sentence_words)
            )

    # Select highest-scoring sentences
    # Orders sentence dictionary keys by their computed scalar density ranks in descending order, slicing the top subset.
    top_sentences = sorted(
        sentence_scores,
        key=sentence_scores.get,
        reverse=True
    )[:num_sentences]

    # Preserve original sentence order for readability
    # Cross-references the targeted array against the original text stream to preserve linear structural narrative flow.
    summary = [
        sentence for sentence in sentences
        if sentence in top_sentences
    ]

    # Stitch the selected array elements back into a uniform space-separated summary paragraph string.
    return " ".join(summary)

In [ ]:
# ── Generate Extractive Summaries ─────────────────────

# Create a separate dataframe for summarisation experiments
# Keep original text so summaries remain human-readable
# Creates a deep copy of specific column subsets to protect the master training dataset from modifications.
df_extractive_summary = df_model[
    [
        "title",
        "abstract",
        "subject_area"
    ]
].copy()

# Generate extractive summaries from the original abstracts
# Each summary keeps the two highest-scoring sentences
# Maps each abstract string dynamically to the scoring algorithm, defaulting to a 2-sentence output constraint.
df_extractive_summary["generated_summary"] = (
    df_extractive_summary["abstract"]
    .apply(lambda x: extractive_summary(x, num_sentences=2))
)

print("Generated summaries successfully.")

# Preview original titles alongside generated summaries
print(
    # Pulls a structural look at the top 5 records to inspect the quality of the shortened text blocks.
    df_extractive_summary[
        [
            "title",
            "generated_summary"
        ]
    ].head()
)

## Experiment and Innovation

In [ ]:
# ── Keyword-Based Title Generation Experiment ───────────
# This is an additional experiment that creates a simple title-like phrase
# from the most frequent informative words in the generated summary.

def generate_keyword_title(text, max_words=8):
    """
    Creates a simple title-like prediction using the
    most frequent informative words in the summary.
    """

    # Return empty output if the input text is invalid
    # Type guard checklist ensuring data pipeline resilience against NaN or numeric artifact noise.
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""

    # Tokenise the summary into lowercase words
    # Converts sentence string structures into distinct lowercase lexical chunks.
    words = word_tokenize(text.lower())

    # Remove stopwords, punctuation and very short tokens
    # Strips out grammatical syntax words, punctuation characters, and filters string token length to avoid fragments.
    filtered_words = [
        word for word in words
        if word not in stop_words
        and word not in string.punctuation
        and word.isalnum()
        and len(word) > 2 # drops 1 or 2 letter connector remnants/abbreviations to prioritize high-value nouns/verbs.
    ]

    # Count word frequencies in the summary
    # Aggregates structural word frequencies into a key-value hash map configuration.
    freq = Counter(filtered_words)

    # Select the most frequent informative words
    # Extracts the specific dictionary keys corresponding to the highest integer counts, up to max_words threshold.
    keywords = [
        word for word, count in freq.most_common(max_words)
    ]

    # Convert selected keywords into title-style capitalisation
    # Concatenates tokens into a string and converts the first character of each word to uppercase using .title().
    return " ".join(keywords).title()


# Generate keyword-based predicted titles from extractive summaries
# Vectorizes the title creation logic across the series containing the newly extracted short summaries.
df_extractive_summary["predicted_title"] = (
    df_extractive_summary["generated_summary"]
    .apply(lambda x: generate_keyword_title(x, max_words=8))
)

print("Predicted titles generated successfully.")

In [ ]:
# ── Example Summarisation and Title Generation Outputs ─────────────

# Display a few example outputs to compare:
# - Original title
# - Original abstract
# - Generated extractive summary
# - Keyword-based predicted title

# Explicit row positions chosen to sample across distinct areas of the dataset.
sample_indices = [0, 25, 100]

for idx in sample_indices:

    # Visual separator line to delineate between different paper samples in the terminal.
    print("=" * 100)

    print("\nORIGINAL TITLE:")
    # Fetches the authentic, author-provided title via integer-location based indexing.
    print(df_extractive_summary["title"].iloc[idx])

    print("\nORIGINAL ABSTRACT:")
    # Outputs the full original text of the research abstract.
    print(df_extractive_summary["abstract"].iloc[idx])

    print("\nGENERATED SUMMARY:")
    # Displays the 2-sentence extractive distillation computed by sentence density weights.
    print(df_extractive_summary["generated_summary"].iloc[idx])

    print("\nKEYWORD-BASED PREDICTED TITLE:")
    # Displays the title-cased string created from the most frequent tokens in the summary.
    print(df_extractive_summary["predicted_title"].iloc[idx])

    # Injects extra spacing between iterations for cleaner text presentation.
    print("\n")

In [ ]:
# ── Summary and Title Length Comparison ───────────────

# Count words in original abstracts, generated summaries and predicted titles
# Approximates word counts across columns by casting values to strings, splitting on whitespace, and taking the length.
df_extractive_summary["abstract_word_count"] = (
    df_extractive_summary["abstract"]
    .apply(lambda x: len(str(x).split()))
)

df_extractive_summary["summary_word_count"] = (
    df_extractive_summary["generated_summary"]
    .apply(lambda x: len(str(x).split()))
)

df_extractive_summary["predicted_title_word_count"] = (
    df_extractive_summary["predicted_title"]
    .apply(lambda x: len(str(x).split()))
)

# Compare average lengths to show how much the text was compressed
# Aggregates each word count column using .mean() and rounds to 2 decimal places for clean reporting.
print(
    "Average abstract length:",
    round(df_extractive_summary["abstract_word_count"].mean(), 2)
)

print(
    "Average summary length:",
    round(df_extractive_summary["summary_word_count"].mean(), 2)
)

print(
    "Average predicted title length:",
    round(df_extractive_summary["predicted_title_word_count"].mean(), 2)
)

In [ ]:
# ── Summary Length Experiment ─────────────────────────

# Test how the extractive summary changes with different sentence lengths
# Defines an iterable sequence to observe how information density expands with higher sentence caps.
summary_lengths = [1, 2, 3]

# Use one original abstract as a sample input
# Isolates the very first abstract row to use as a static control variable across all loops.
sample_text = df_extractive_summary["abstract"].iloc[0]

for n in summary_lengths:

    # Iterative visual break to easily separate lengths in the execution window.
    print("=" * 80)

    print(f"\nSUMMARY USING {n} SENTENCE(S):\n")

    # Generate summary with selected number of sentences
    # Invokes the scoring algorithm, explicitly passing the current loop's integer as the extraction ceiling.
    summary = extractive_summary(
        sample_text,
        num_sentences=n
    )

    print(summary)

    print("\n")

# WEEK 3 COURSEWORK

## LLM Setup

In [ ]:
# ── Week 3: Central LLM API Helper ─────────────────────────────
# Store Groq API key for all LLM-based tasks
# Note: Hardcoding API keys directly in source code is a major security risk. 
# Consider migrating this to an environment variable or a secret management tool.
GROQ_API_KEY = "YOUR_GROQ_API_KEY"


# ── Delay Helper ───────────────────────────────────────────────

# Small reusable delay function to reduce rapid API requests
def api_delay(seconds=2):
    # Suspends execution of the current thread to respect rate limits and pacing strategies.
    time.sleep(seconds)


# ── Main LLM Response Function ─────────────────────────────────

def llm_response(prompt, max_retries=5):
    """
    Sends a prompt to the LLM and returns the generated response.

    Reused for:
    - LLM classification
    - LLM summarisation
    - LLM title generation
    - RAG-enhanced prompting

    Includes retry handling for temporary rate limits.
    """

    # Groq OpenAI-compatible endpoint
    # Base destination URI matching the OpenAI REST structure for chat inference engines.
    url = "https://api.groq.com/openai/v1/chat/completions"

    # Authentication and request headers
    # Sets bearer authentication tokens and informs the server to expect JSON-formatted metadata payloads.
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    # Main request payload
    # Structured payload mapping the text instructions, runtime hyper-parameters, and model target.
    payload = {
        "model": "llama-3.1-8b-instant", # Target inference engine.
        "messages": [
            {
                "role": "user",
                "content": prompt     # The structural instructions passed directly to the LLM context.
            }
        ],

        # Lower temperature gives more consistent outputs
        "temperature": 0,             # Deterministic sampling mode. Guarantees predictable, reproducible text choices.

        # Limit response length
        "max_tokens": 150             # Hard boundary to control cost and avoid excessive text generation.
    }

    # Retry loop for temporary API rate-limit issues
    # Linear exponential back-off loop designed to self-heal when hitting high-volume API restrictions.
    for attempt in range(max_retries):

        # Sends a blocking HTTP POST request to the remote inference server.
        response = requests.post(
            url,
            json=payload,
            headers=headers
        )

        # Successful API response
        # HTTP 200 OK block indicating that the generation task completed successfully.
        if response.status_code == 200:
            return (
                # Drill down through JSON nested fields to isolate the generation text segment, stripping surrounding whitespace.
                response.json()["choices"][0]["message"]["content"]
                .strip()
            )

        # Handle rate-limit errors gracefully
        # Checks for HTTP 429 Too Many Requests status code indicating structural usage limits have been reached.
        elif response.status_code == 429:

            # Implements an additive delay sequence that expands further with each consecutive failure.
            wait_time = 10 * (attempt + 1)

            print(f"Rate limit hit. Waiting {wait_time} seconds...")

            # Halts code execution for the progressive back-off penalty duration before attempting a retry.
            api_delay(wait_time)

        # Return any other API errors directly
        # Intercepts other structural failures (e.g., 401 Unauthorized, 404 Not Found) and returns them immediately.
        else:
            return f"Error {response.status_code}: {response.text}"

    # Final fallback if retries continue failing
    # Returns a failure string if all retry loop budget instances are exhausted without an HTTP 200 resolution.
    return "Error 429: Rate limit after retries"

## Step 1: LLM-Based Text Classification

In [ ]:
# ── Prompt 1: Basic Classification Prompt ──────────────────────
def classify_basic_prompt(abstract):
    """
    Basic LLM classification prompt.

    This is used as a simple baseline so it can be compared
    against a more structured prompt.
    """

    # Ask the LLM to classify the abstract into one broad subject area
    # Constructs a simple zero-shot instruction prompt with an explicit, hardcoded list of target labels.
    prompt = f"""
    Classify the following scientific paper abstract
    into ONE category only.

    Categories:
    - cs
    - math
    - physics
    - q-bio
    - q-fin
    - stat

    Abstract:
    {abstract}

    Only output the category label.
    """

    # Passes the multi-line string instruction to the centralized Groq/Llama API client.
    return llm_response(prompt)

In [ ]:
# ── Prompt 2: Structured Classification Prompt ────────────────
def classify_structured_prompt(abstract):
    """
    Structured LLM classification prompt.

    This version gives clearer category descriptions to test whether
    prompt engineering improves classification performance.
    """

    # Provide clearer label definitions and stricter task instructions
    # Uses role prompting ("You are...") and provides semantic definitions for each class 
    # to ground the model's domain knowledge and limit out-of-vocabulary label generation.
    prompt = f"""
    You are classifying scientific papers by subject area.

    Read the abstract carefully and choose exactly ONE label.

    Allowed labels:
    - cs: computer science, AI, algorithms, systems
    - math: mathematics, proofs, algebra, geometry, analysis
    - physics: physics, quantum, astronomy, particles, materials
    - q-bio: biology, genomics, neuroscience, bioinformatics
    - q-fin: finance, markets, trading, risk modelling
    - stat: statistics, probability, inference, data analysis

    Abstract:
    {abstract}

    Only output the category label.
    """

    # Passes the highly engineered instruction context to the remote inference handler.
    return llm_response(prompt)

In [ ]:
def clean_llm_label(raw_response):
    """
    Converts LLM output into one valid category label using exact word boundary matching.
    Prevents words inside sentences (like 'physics' or 'cs') from cross-contaminating.
    """
    import re
    
    # Casts raw response to string, strips external whitespaces, and normalizes casing to lowercase.
    cleaned = str(raw_response).strip().lower()
    
    # Defines the explicit collection of target categorical identifiers expected by the system.
    valid_labels = ["cs", "math", "physics", "q-bio", "q-fin", "stat"]
    
    # 1. First check if the raw output is EXACTLY just the label (best case)
    # Short-circuits the evaluation loop if the model successfully adhered to the formatting constraints.
    if cleaned in valid_labels:
        return cleaned
        
    # 2. If it's embedded in a sentence, look for the standalone word
    # Fallback parser to extract targets out of conversational chatter or full-sentence outputs.
    for label in valid_labels:
        # \b ensures we match 'cs' as a word, not as part of 'physics' or 'discussion'
        # re.escape guarantees special characters (like the hyphen in q-bio) are treated as literal text.
        if re.search(r'\b' + re.escape(label) + r'\b', cleaned):
            return label
            
    # Returns a safe default string if no allowed keyword targets match the boundary criteria.
    return "unknown"

In [ ]:
# ── Stratified LLM Classification Evaluation ───────────────────

# Create a balanced evaluation subset with 5 samples from each subject area
# Groups the master DataFrame by its target class to prevent majority-class bias during evaluation.
llm_classification_df = (
    df_model
    .groupby("subject_area", group_keys=False)
    .apply(
        # Draws exactly 5 random records from each group using a fixed random seed for reproducibility.
        lambda x: x.sample(
            min(len(x), 5),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

# Run basic prompt classification and clean the returned labels
# Cascades the abstract text through the simple prompt API, then passes the text response through the word-boundary cleaner.
llm_classification_df["llm_prediction_basic"] = (
    llm_classification_df["abstract"]
    .apply(classify_basic_prompt)
    .apply(clean_llm_label)
)

# Run structured prompt classification and clean the returned labels
# Cascades the abstract text through the engineered prompt API, then passes the text response through the word-boundary cleaner.
llm_classification_df["llm_prediction_structured"] = (
    llm_classification_df["abstract"]
    .apply(classify_structured_prompt)
    .apply(clean_llm_label)
)

# Compare true labels against predictions from both prompts
print(
    # Pulls a side-by-side comparison table of ground-truth classes versus basic and structured LLM outputs.
    llm_classification_df[
        [
            "subject_area",
            "llm_prediction_basic",
            "llm_prediction_structured"
        ]
    ]
)

In [ ]:
# ── Evaluate Basic vs Structured Prompt Performance ────────────

# Compare both prompting styles using the same balanced evaluation sample
# Maps human-readable experiment names to their respective prediction column identifiers.
prompt_results = {
    "Basic Prompt": "llm_prediction_basic",
    "Structured Prompt": "llm_prediction_structured"
}

for prompt_name, prediction_column in prompt_results.items():

    # Compute overall classification accuracy
    # Calculates the global accuracy ratio by matching true labels against the active prompt column.
    accuracy = accuracy_score(
        llm_classification_df["subject_area"],
        llm_classification_df[prediction_column]
    )

    print("=" * 80)
    print(f"{prompt_name} Accuracy: {accuracy:.2f}")

    # Display per-class precision, recall and F1 score
    # Outputs a comprehensive text breakdown of class-level metrics.
    print(
        classification_report(
            llm_classification_df["subject_area"],
            llm_classification_df[prediction_column],
            zero_division=0 # Prevents runtime warnings by setting metrics to 0 if a class has no predicted samples.
        )
    )

# Step 2: LLM-Based Summarisation

In [ ]:
# ── LLM Summary and Title Generation Function ──────────────────

def summarise_and_generate_title(abstract):
    """
    Uses the LLM to generate:
    - a short factual summary
    - a possible academic paper title

    This provides an abstractive LLM-based comparison to the
    earlier extractive summarisation method.
    """

    # Prompt the LLM to summarise the abstract and generate a specific title
    # Constructs a multi-task prompt combining abstractive generation with explicit negative constraints 
    # and a strict output template to facilitate clean string parsing later.
    prompt = f"""
    Read the scientific abstract carefully.

    Your tasks:
    1. Write a factual 2-sentence summary.
    2. Generate a possible paper title that is close to the wording and topic of the abstract.

    Rules:
    - Do not make the title broader than the abstract.
    - Do not add concepts that are not clearly present in the abstract.
    - Keep the generated title short, specific, and academic.
    - Avoid phrases like "applications", "connections", "role of", or "towards"
      unless they are clearly central.

    Abstract:
    {abstract}

    Output exactly as:

    Summary: <2 sentence summary>
    Generated Title: <short specific title>
    """

    # Send prompt to the central LLM helper
    # Returns the raw completion string following the requested formatting scheme.
    return llm_response(prompt)

In [ ]:
# ── Test LLM Summarisation and Title Generation ────────────────

# Use a small sample because each abstract requires an LLM API call
# Original abstracts are used because LLM summaries should remain readable and natural
# Creates a random 10-row slice of the dataframe to evaluate without hitting aggressive API rate limits.
llm_summary_df = df_model.sample(
    10,
    random_state=42
).copy()

# Generate LLM summary and possible title for each abstract
# Iterates sequentially over the sampled rows, wrapping the loop in a tqdm progress bar 
# to monitor generation velocity and API response latency.
llm_summary_df["llm_summary_title_output"] = [
    summarise_and_generate_title(abstract)
    for abstract in tqdm(llm_summary_df["abstract"])
]

# Preview original title, abstract and generated LLM output
# Pulls a side-by-side view of the top 5 rows to visually inspect the raw model generation against the source texts.
llm_summary_df[
    [
        "title",
        "abstract",
        "llm_summary_title_output"
    ]
].head()

In [ ]:
# ── Extract LLM Summary and Generated Title ────────────────────

def extract_summary(output):
    """
    Extracts only the summary section from the LLM response.
    """

    # Forces the input object into a string format to handle potential missing or NaN structures safely.
    output = str(output)

    # Extract text between "Summary:" and "Generated Title:"
    # Validates that the expected anchor keys from the prompt engineering template exist in the text block.
    if "Summary:" in output and "Generated Title:" in output:

        return (
            output
            # Isolates the front section of the string by splitting on the trailing title anchor header.
            .split("Generated Title:")[0]
            # Erases the leading structural "Summary:" text identifier marker.
            .replace("Summary:", "")
            # Strips out rogue hanging carriage returns, line breaks, or blank space margins.
            .strip()
        )

    # Fallback if formatting differs
    # Returns the unaltered raw output payload intact if the token delimiters fail to match.
    return output


def extract_generated_title(output):
    """
    Extracts only the generated title from the LLM response.
    """

    # Ensures uniform string attributes are available for slicing operations.
    output = str(output)

    # Extract everything after "Generated Title:"
    # Confirms the exact header key is present before attempting an array split.
    if "Generated Title:" in output:

        title = (
            output
            # Grabs the last split index array element containing everything following the title header anchor.
            .split("Generated Title:")[-1]
            .strip()
        )

        # Remove quotation marks if present
        # Cleans out any extraneous literal quote markers injected into the academic string by the LLM.
        title = title.replace('"', '')

        return title

    # Fallback if title extraction fails
    # Returns an explicit error placeholder string instead of breaking the mapping loop sequence.
    return "Not extracted"


# Extract clean summary text from the combined LLM output
# Maps the custom slicing logic over the unstructured string payload column to construct a dedicated text summary series.
llm_summary_df["llm_summary"] = (
    llm_summary_df["llm_summary_title_output"]
    .apply(extract_summary)
)

# Extract generated title from the combined LLM output
# Maps the targeted trailing-edge text extraction parser to populating clean structural academic titles.
llm_summary_df["generated_title"] = (
    llm_summary_df["llm_summary_title_output"]
    .apply(extract_generated_title)
)

# Preview original title, extracted summary and generated title
# Filters the resulting testing slice DataFrame to visually compare the human-authored benchmarks against the clean extracted fields.
llm_summary_df[
    [
        "title",
        "llm_summary",
        "generated_title"
    ]
]

In [ ]:
# ── Lexical and Semantic Similarity Evaluation ─────────────────

# TF-IDF similarity measures lexical word overlap between titles
def tfidf_title_similarity(original_title, generated_title):
    """
    Computes a bag-of-words similarity metric based on term frequency overlap.
    """
    # Converts inputs into lowercased strings to ensure uniform word tokenization.
    texts = [
        str(original_title).lower(),
        str(generated_title).lower()
    ]

    # Dynamically builds a temporary text-vectorization pipeline stripping common English stop words.
    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    # Fits vocabulary over both titles and returns their matching sparse TF-IDF feature matrices.
    tfidf_matrix = vectorizer.fit_transform(texts)

    # Computes the cosine angle between the two sparse term vectors to calculate a score from 0.0 to 1.0.
    similarity = cosine_similarity(
        tfidf_matrix[0:1],
        tfidf_matrix[1:2]
    )[0][0]

    return similarity


# Sequence similarity measures character-level similarity
def sequence_title_similarity(original_title, generated_title):
    """
    Measures the structural edit-distance ratio using the Gestalt Pattern Matching algorithm.
    """
    # Computes string closeness based on the longest common contiguous character subsequences.
    return SequenceMatcher(
        None,
        str(original_title).lower(),
        str(generated_title).lower()
    ).ratio()


# Semantic similarity measures meaning-level similarity using embeddings
def semantic_title_similarity(original_title, generated_title):
    """
    Measures conceptual overlap by mapping strings into a deep semantic vector space.
    """
    # Converts the raw textual inputs into high-dimensional dense vector representations via a bi-encoder model.
    embeddings = embedding_model.encode(
        [
            str(original_title),
            str(generated_title)
        ]
    )

    # Computes the cosine similarity between the dense embeddings to evaluate proximity of abstract meaning.
    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    return similarity


# Compute TF-IDF lexical similarity scores
# Iterates row-wise over the slice to map statistical keyword overlap metrics across matching title pairs.
llm_summary_df["tfidf_similarity_score"] = (
    llm_summary_df.apply(
        lambda row: tfidf_title_similarity(
            row["title"],
            row["generated_title"]
        ),
        axis=1
    )
)

# Compute sequence similarity scores
# Evaluates literal character-matching continuity to detect structural string rearrangements or typos.
llm_summary_df["sequence_similarity_score"] = (
    llm_summary_df.apply(
        lambda row: sequence_title_similarity(
            row["title"],
            row["generated_title"]
        ),
        axis=1
    )
)

# Compute semantic similarity scores using embeddings
# Captures conceptual alignment, allowing the system to reward true contextual matches even when different words are used.
llm_summary_df["semantic_similarity_score"] = (
    llm_summary_df.apply(
        lambda row: semantic_title_similarity(
            row["title"],
            row["generated_title"]
        ),
        axis=1
    )
)

# Compare original titles with generated titles and similarity metrics
# Displays a unified analytical summary table to validate and compare the three distinct evaluation strategies.
llm_summary_df[
    [
        "title",
        "generated_title",
        "tfidf_similarity_score",
        "sequence_similarity_score",
        "semantic_similarity_score"
    ]
]

In [ ]:
# ── Average Similarity Scores ──────────────────────────────────

# Calculate average lexical, sequence and semantic similarity scores
# Aggregates performance across the evaluation slice by calculating the arithmetic mean for each metric.
average_scores = llm_summary_df[
    [
        "tfidf_similarity_score",
        "sequence_similarity_score",
        "semantic_similarity_score"
    ]
].mean()

print("Average Similarity Scores:\n")

print(
    f"TF-IDF Similarity: "
    f"{average_scores['tfidf_similarity_score']:.2f}"
)

print(
    f"Sequence Similarity: "
    f"{average_scores['sequence_similarity_score']:.2f}"
)

print(
    f"Semantic Similarity: "
    f"{average_scores['semantic_similarity_score']:.2f}"
)

In [ ]:
# ── Display Full LLM Summary and Title Examples ────────────────

# Display a few complete examples for qualitative evaluation
# This helps compare generated titles and summaries against original titles

# Iterates over the first five rows of the evaluation dataframe as index-row pairs for granular debugging.
for i, row in llm_summary_df.head(5).iterrows():

    # Long visual banner to cleanly segment each paper's comparative data profile.
    print("=" * 100)

    print("ORIGINAL TITLE:")
    # Displays the reference benchmark title written by the authors.
    print(row["title"])

    print("\nGENERATED TITLE:")
    # Displays the zero-shot academic title synthesized by the model.
    print(row["generated_title"])

    print("\nLLM SUMMARY:")
    # Displays the abstractive two-sentence factual summary text.
    print(row["llm_summary"])

    print("\nTF-IDF SIMILARITY SCORE:")
    # Outputs the token-overlap score, rounded to 2 decimal places.
    print(round(row["tfidf_similarity_score"], 2))

    print("\nSEQUENCE SIMILARITY SCORE:")
    # Outputs the character-matching edit-distance ratio, rounded to 2 decimal places.
    print(round(row["sequence_similarity_score"], 2))

    print("\nSEMANTIC SIMILARITY SCORE:")
    # Outputs the deep-learning vector similarity score, rounded to 2 decimal places.
    print(round(row["semantic_similarity_score"], 2))

# Step 3: Retrieval-Augmented Generation (RAG) Classification

In [ ]:
# ── Reverse Label Mapping ──────────────────────────────────────

# Convert encoded numeric labels back into readable category names
# Inverts the original string-to-integer dictionary, transforming encoded IDs back into human-readable text labels.
label_mapping = {
    encoded: label
    for label, encoded in mapping.items()
}

In [ ]:
# ── Build Semantic Vector Database for RAG ─────────────────────

# Load sentence embedding model for semantic retrieval
# Initializes a lightweight, 384-dimensional bi-encoder model optimized for semantic text similarity.
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

# Use training abstracts only to avoid test-set leakage
# Converts the training text series and corresponding ground-truth arrays into flat Python lists.
rag_documents = X_abs_train.tolist()
rag_labels = y_train.tolist()

print(f"Encoding {len(rag_documents):,} training abstracts...")

# Encode training abstracts into dense semantic vectors
# Transforms raw text chunks into dense numeric arrays, processing them in chunks of 256 for optimal GPU memory usage.
rag_doc_vectors = semantic_model.encode(
    rag_documents,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True # Forces the output array format to standard NumPy matrices for easier indexing downstream.
)

print(f"Vector database shape: {rag_doc_vectors.shape}")

In [ ]:
# ── MMR Retrieval Function ─────────────────────────────────────

def mmr_retrieval(query_vector, doc_vectors, top_k=4, lambda_param=0.5):
    """
    Retrieves examples using Maximal Marginal Relevance.

    This balances:
    - relevance to the query abstract
    - diversity among retrieved examples
    """

    # Enforces a 2D matrix shape on the query vector to make it compatible for matrix multiplication / cosine calculations.
    query_vector = query_vector.reshape(1, -1)
    
    # Computes a flat 1D array of cosine similarity scores between the query and all database vectors.
    query_sims = cosine_similarity(query_vector, doc_vectors)[0]

    selected = []
    candidate_ids = list(range(len(doc_vectors)))

    # Iterative selection loop running until the desired top_k document quota is fulfilled.
    for _ in range(top_k):

        if not selected:
            # First pass shortcut: grabs the single most similar document based entirely on raw relevance scores.
            best = int(np.argmax(query_sims))

            selected.append(best)
            candidate_ids.remove(best)

            continue

        # Gathers all previously selected dense vectors to compute redundancy comparisons.
        selected_vectors = doc_vectors[selected]
        mmr_scores = []

        # Evaluate every remaining candidate index against the MMR formula.
        for idx in candidate_ids:

            # Pure semantic similarity score relative to the user's input query.
            relevance = query_sims[idx]

            # Finds the maximum similarity score between the current candidate and any already-selected document.
            redundancy = cosine_similarity(
                doc_vectors[idx].reshape(1, -1),
                selected_vectors
            ).max()

            # The core MMR objective function: penalizes redundant semantic content using the lambda weight.
            score = (
                lambda_param * relevance
                - (1 - lambda_param) * redundancy
            )

            mmr_scores.append((idx, score))

        # Identifies the document identifier that yields the highest calculated MMR score.
        best = max(mmr_scores, key=lambda x: x[1])[0]

        selected.append(best)
        candidate_ids.remove(best)

    # Returns the list of chosen document indices representing an optimized balance of accuracy and variety.
    return selected

In [ ]:
# ── Retrieve Similar Labelled Examples ─────────────────────────

def retrieve_similar_examples(query_abstract, top_k=4, lambda_param=0.5):
    """
    Retrieves semantically similar training abstracts and their labels.
    """

    # Encode query abstract
    # Generates a dense vector embedding for the incoming target abstract, isolating the first row array.
    query_vector = semantic_model.encode(
        [query_abstract],
        convert_to_numpy=True
    )[0]

    # Retrieve diverse semantic neighbours
    # Executes the Maximal Marginal Relevance algorithm to extract a balanced, non-redundant set of document indices.
    top_indices = mmr_retrieval(
        query_vector,
        rag_doc_vectors,
        top_k=top_k,
        lambda_param=lambda_param
    )

    # Calculate similarity scores for retrieved examples
    # Computes individual proximity scores between the query vector and the specific MMR-selected subset.
    similarities = cosine_similarity(
        query_vector.reshape(1, -1),
        rag_doc_vectors[top_indices]
    )[0]

    retrieved_examples = []

    # Construct the structural reference context payload for few-shot prompting injection
    for rank, idx in enumerate(top_indices):

        retrieved_examples.append({
            "abstract": rag_documents[idx], # The training text block to serve as an in-context exemplar.
            "label": rag_labels[idx],       # The target category integer or string label for the example.
            "similarity": float(similarities[rank]) # Logged for routing diagnostics or debugging trace telemetry.
        })

    return retrieved_examples

In [ ]:
# ── Test RAG Retrieval ─────────────────────────────────────────

# Use one test abstract to check whether retrieval looks sensible
# Extracts the first record from the holdout testing set to serve as an evaluation query.
query_abstract = X_abs_test.iloc[0]

# Triggers the semantic-search pipeline to retrieve up to 4 diverse reference examples.
retrieved_examples = retrieve_similar_examples(
    query_abstract,
    top_k=4
)

print("Query abstract:")
# Prints a truncated 300-character preview of the validation target query to save console room.
print(query_abstract[:300])

print("=" * 80)

# Unpacks and prints details for each matching context document discovered by the vector index.
for i, example in enumerate(retrieved_examples, 1):

    print(f"Retrieved Example {i}")
    # Displays the raw cosine closeness score relative to the query array.
    print(f"Similarity: {round(example['similarity'], 4)}")
    # Evaluates the internal label ID against our label dictionary to map it back to a readable string (e.g., 'math').
    print(f"Label: {label_mapping[example['label']]}")
    # Displays a truncated preview of the training text block serving as the few-shot template context.
    print(f"Abstract: {example['abstract'][:300]}")
    print("=" * 80)

In [ ]:
# ── Semantic RAG Classification Function ───────────────────────

def classify_rag_semantic(abstract, top_k=4):
    """
    Classifies an abstract using retrieved labelled examples as context.
    """

    # Gathers dynamic, high-relevance few-shot examples using MMR-driven vector search.
    examples = retrieve_similar_examples(
        abstract,
        top_k=top_k
    )

    example_text = ""

    # Iterates over the retrieved set to build a formatted string block of in-context exemplars.
    for i, example in enumerate(examples, 1):

        # Converts numerical database tags back into string-based category names (e.g., 'cs').
        category_name = label_mapping[
            example["label"]
        ]

        # Injects the structured textual sample block directly into the accumulator.
        example_text += f"""
Example {i}

Abstract:
{example["abstract"]}

Category:
{category_name}
"""

    # Dynamically injects the historical examples block along with the new target query abstract.
    prompt = f"""
    You are classifying scientific paper abstracts.

    Use the similar labelled examples below as reference.

    Similar labelled examples:
    {example_text}

    Now classify the following abstract into exactly ONE category.

    Categories:
    - cs
    - math
    - physics
    - q-bio
    - q-fin
    - stat

    Abstract:
    {abstract}

    Only output the category label.
    """

    # Dispatches the unified context prompt to the remote LLM microservice.
    raw_output = llm_response(prompt)

    # Standardizes the raw response via boundary-matching regex cleanup before final routing.
    return clean_llm_label(raw_output)

In [ ]:
# ── Compare Structured Prompt vs Semantic RAG ──────────────────

# Number of sample records to isolate from the evaluation split.
sample_size = 10

# Draws a random subset of validation abstracts to benchmark zero-shot vs few-shot (RAG) approaches.
rag_eval_sample = X_abs_test.sample(
    n=sample_size,
    random_state=42
)

# Extracts the corresponding ground-truth encoded label indices using the sampled text index keys.
rag_eval_labels = y_test.loc[
    rag_eval_sample.index
]

# Trackers for statistical downstream evaluation arrays.
true_labels = []
structured_preds = []
semantic_rag_preds = []

# Iterates sequentially over the query items to feed both competing inference pathways.
for i, (abstract, true_encoded) in enumerate(
    zip(rag_eval_sample, rag_eval_labels),
    1
):

    print(f"[{i}/{sample_size}] Classifying...")

    # Translates the active row's numeric class marker to its human-readable category name.
    true_labels.append(
        label_mapping[true_encoded]
    )

    # Pathway A: Generates a zero-shot inference prediction based purely on structured text instructions.
    structured_preds.append(
        clean_llm_label(
            classify_structured_prompt(abstract)
        )
    )

    # Pathway B: Generates a dynamic few-shot inference prediction enriched with real-time vector search dependencies.
    semantic_rag_preds.append(
        classify_rag_semantic(abstract)
    )

print("RAG comparison complete.")

In [ ]:
# ── RAG Results Table ──────────────────────────────────────────

# Constructs a unified validation DataFrame combining true categories against both classification methods.
rag_results_df = pd.DataFrame({
    "True Label": true_labels,                # Ground-truth human-annotated category markers.
    "Structured Prompt": structured_preds,    # Predictions generated via zero-shot domain role prompts.
    "Semantic RAG": semantic_rag_preds        # Predictions generated via dynamic dynamic few-shot MMR retrieval.
})

# Displays the tabular evaluation matrix for clean side-by-side comparative diagnostics.
rag_results_df

In [ ]:
# ── RAG Accuracy Summary ───────────────────────────────────────

# Initialize a mutable list structure to collect descriptive stats per strategy.
rag_summary_rows = []

# Iterates over each distinct prompting pipeline column to isolate performance characteristics.
for method in [
    "Structured Prompt",
    "Semantic RAG"
]:

    # Determines the total number of validation iterations evaluated in the comparison block.
    total = len(rag_results_df)

    # Computes the total number of exact token matches relative to the true category criteria.
    correct = (
        rag_results_df["True Label"]
        == rag_results_df[method]
    ).sum()

    # Tracks instances where the LLM violated parsing formatting rules and returned an 'unknown' flag.
    unknown = (
        rag_results_df[method]
        == "unknown"
    ).sum()

    # Appends the calculated quantitative data row for the structural summary sheet matrix.
    rag_summary_rows.append({
        "Method": method,
        "Accuracy": round(correct / total, 3), # Global evaluation precision ratio, rounded to 3 decimal places.
        "Correct": correct,
        "Wrong": total - correct - unknown,    # Deducts valid errors from parsing failures to isolate true logical mismatches.
        "Unknown/Unparsed": unknown,           # Captures validation fallback metrics for structural analysis.
        "Total": total
    })

# Converts the aggregated dictionary rows into a formalized Pandas dataframe summary profile.
rag_summary_df = pd.DataFrame(rag_summary_rows)

# Displays the structural breakdown table comparing zero-shot and dynamic in-context retrieval performance.
rag_summary_df

# Step 4: Comparison with Classical Machine Learning

In [ ]:
# ── Step 4a: Classical ML vs LLM Classification Comparison ─────

# Select best classical ML pipeline from previous ML results
# Extracts the top-performing model architecture and its corresponding text feature representation configuration.
best_ml_row = ml_results_df.iloc[0]

classical_method = best_ml_row["Model"]
classical_config = best_ml_row["Configuration"]

best_model = models[classical_method]
best_config = feature_configs[classical_config]

# Train best classical model
# Fits the selected traditional machine learning estimator on the isolated feature-engineered training matrix.
best_model.fit(
    best_config["X_train"],
    y_train
)

# Create reverse label mapping from LabelEncoder
# Dynamically reconstructs an integer-to-string dictionary using the encoder's internal class list.
reverse_mapping = {
    i: label
    for i, label in enumerate(le.classes_)
}

# Use same test sample for both Classical ML and LLM
# Isolates a locked 20-record slice to ensure a perfectly fair evaluation benchmark across paradigms.
sample_size = 20

comparison_sample = X_abs_test.sample(
    n=sample_size,
    random_state=42
)

comparison_labels = y_test.loc[
    comparison_sample.index
]

# Translates the target validation class series back to their true textual names.
true_labels = [
    reverse_mapping[label]
    for label in comparison_labels
]

# Get matching row positions from X_abs_test
# Maps unique pandas DataFrame index keys to their relative row positions for exact coordinate slicing of NumPy arrays.
sample_positions = [
    X_abs_test.index.get_loc(idx)
    for idx in comparison_sample.index
]

# Classical ML predictions on same sample
# Scores the classical model using matching sparse matrix or dense embedding row slices from the test matrix.
ml_predictions_encoded = best_model.predict(
    best_config["X_test"][sample_positions]
)

ml_predictions = [
    reverse_mapping[label]
    for label in ml_predictions_encoded
]

# LLM predictions on same sample
# Sequentially routes the identical sample text blocks through the structured zero-shot instruction prompt.
llm_predictions = []

for abstract in comparison_sample:

    output = classify_structured_prompt(
        abstract
    )

    llm_predictions.append(
        clean_llm_label(output)
    )

# Calculate metrics on the same sample
# Computes the raw percentage of correctly matched classes over the 20-row sample for the classical pipeline.
classical_sample_accuracy = accuracy_score(
    true_labels,
    ml_predictions
)

# Evaluates structural macro F1-score to check performance balance across minority classes.
classical_sample_macro_f1 = f1_score(
    true_labels,
    ml_predictions,
    average="macro"
)

# Computes performance statistics over the sample slice for the zero-shot LLM pathway.
llm_accuracy = accuracy_score(
    true_labels,
    llm_predictions
)

llm_macro_f1 = f1_score(
    true_labels,
    llm_predictions,
    average="macro"
)

# Create comparison table
# Organizes the aggregated evaluation statistics into a structural comparison frame.
classification_results = pd.DataFrame({
    "Pipeline": [
        "Classical ML",
        "LLM"
    ],

    "Method": [
        f"{classical_method} | {classical_config}",
        "Structured Prompt Classification"
    ],

    "Accuracy": [
        round(classical_sample_accuracy, 3),
        round(llm_accuracy, 3)
    ],

    "Macro F1": [
        round(classical_sample_macro_f1, 3),
        round(llm_macro_f1, 3)
    ]
})

display(classification_results)

# Show example predictions
# Isorates a minor 5-row lookahead table to inspect actual prediction labels against true grounds.
classification_examples = pd.DataFrame({
    "True Label": true_labels[:5],
    "Classical ML Prediction": ml_predictions[:5],
    "LLM Prediction": llm_predictions[:5]
})

display(classification_examples)

# Visual comparison
# Prepares dimensions and metric sets for an analytical side-by-side performance bar plot.
metrics = [
    "Accuracy",
    "Macro F1"
]

classical_scores = [
    classical_sample_accuracy,
    classical_sample_macro_f1
]

llm_scores = [
    llm_accuracy,
    llm_macro_f1
]

x = range(len(metrics))

plt.figure(figsize=(8, 5))

# Renders the traditional ML scores shifted slightly leftward.
plt.bar(
    [i - 0.2 for i in x],
    classical_scores,
    width=0.4,
    label="Classical ML"
)

# Renders the LLM scores shifted slightly rightward.
plt.bar(
    [i + 0.2 for i in x],
    llm_scores,
    width=0.4,
    label="LLM"
)

# Sets visual structure, formatting, and text axis labels before generating the plot layout.
plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Classification Performance Comparison")
plt.legend()
plt.show()

In [ ]:
# ── Step 4b: Summarisation Quality – Classical vs LLM ──────────

# Use one paper from the LLM summary sample for direct comparison
# Grabs the first item from the pre-generated evaluation slice to build a focused case study.
summary_example = llm_summary_df.iloc[0]

original_title = summary_example["title"]
original_abstract = summary_example["abstract"]

# Generate classical extractive summary using the Week 2 function
# Uses a graph-based or frequency-based heuristic to pull the top 2 highest-ranked raw sentences directly.
classical_summary = extractive_summary(
    original_abstract,
    num_sentences=2
)

# Generate LLM summary and possible title
# Requests a joint generation payload containing both abstractive text and a specialized title.
llm_summary_output = summarise_and_generate_title(
    original_abstract
)

# Extract only the LLM summary from the combined output
# Strips the structural title keys away to isolate only the 2-sentence generated summary paragraph.
llm_summary_only = extract_summary(
    llm_summary_output
)

# ── Example Output ─────────────────────────────────────────────

print("=" * 100)
print("SUMMARISATION EXAMPLE")
print("=" * 100)

print("\nORIGINAL TITLE:")
print(original_title)

print("\nCLASSICAL / EXTRACTIVE SUMMARY:")
# Prints the untouched, verbatim sentence selections extracted from the source paper.
print(classical_summary)

print("\nLLM SUMMARY:")
# Prints the clean, paraphrased text block generated by the model.
print(llm_summary_only)

print("\nFULL LLM OUTPUT:")
# Prints the raw, unparsed multi-line string containing headers and markdown sequences.
print(llm_summary_output)

# ── Summarisation Comparison Table ─────────────────────────────

# Compare the behaviour of classical and LLM-based summarisation
# Maps the qualitative architectural differences between extractive indexing and generative synthesis.
summarisation_results = pd.DataFrame({
    "Criteria": [
        "Summary approach",
        "Output generation",
        "Context handling",
        "Paraphrasing",
        "Title generation",
        "Determinism"
    ],

    "Classical / Extractive Method": [
        "Extractive",
        "Uses original sentences from the abstract",
        "Limited to sentence selection",
        "Minimal",
        "Not directly supported",
        "Deterministic" # Running the algorithm repeatedly on identical text will always return the same indices.
    ],

    "LLM-Based Method": [
        "Abstractive",
        "Generates summary from semantic meaning",
        "Uses broader contextual understanding",
        "Supports paraphrasing",
        "Supported",
        "Outputs may vary between runs" # Subject to stochastic temperature sampling and token probability ranges.
    ]
})

display(summarisation_results)

# ── Summary Length Visual ──────────────────────────────────────

# Compare summary lengths using word counts
# Computes rough word counts by breaking strings along whitespace boundaries.
classical_length = len(classical_summary.split())
llm_length = len(llm_summary_only.split())

methods = [
    "Classical / Extractive",
    "LLM Summary"
]

lengths = [
    classical_length,
    llm_length
]

plt.figure(figsize=(8, 4))

# Renders a side-by-side vertical bar chart evaluating generation density and brevity.
plt.bar(methods, lengths)

plt.ylabel("Word Count")
plt.title("Summary Length Comparison")

plt.show()

# WEEK 4 COURSEWORK

In [ ]:
# ── Best Classical ML Pipeline ─────────────────────────────────

# Select the best-performing classical ML pipeline
# Identifies the highest-ranked classical model configuration based on the initial benchmarking.
best_ml_row = ml_results_df.iloc[0]

classical_method = best_ml_row["Model"]
classical_config = best_ml_row["Configuration"]

best_model = models[classical_method]
best_config = feature_configs[classical_config]

# Train the best model on the training split
# Fits the chosen estimator using its corresponding vectorizer/feature split data matrix.
best_model.fit(
    best_config["X_train"],
    y_train
)

# Reverse label mapping for readable predictions
# Reconstructs an index-to-string dictionary from LabelEncoder to decode numerical targets into category names.
reverse_mapping = {
    i: label
    for i, label in enumerate(le.classes_)
}

In [ ]:
# ── 5c: Main NLP Pipeline ──────────────────────────────────────

def analyse_abstract(user_abstract, task_choice):
    """
    Runs the selected NLP task on a user-provided abstract.

    Supported tasks:
    - Classical ML classification
    - LLM classification
    - Extractive summarisation
    - LLM summary and title generation
    """

    try:

        # Check that the user entered enough text for meaningful analysis
        # Validates minimal string length thresholds to protect vectorizers and LLM context performance.
        if not user_abstract or len(user_abstract.strip()) < 30:

            return (
                "Please enter a longer scientific abstract.",
                "",
                "",
                "",
                ""
            )

        # Empty defaults allow task-specific outputs
        # Pre-allocates return value strings so unselected tasks gracefully return empty values.
        ml_prediction = ""
        llm_prediction = ""
        classical_summary = ""
        summary = ""
        title = ""

        # Transform user input using the feature configuration of the best ML model
        # Selects and fits text shapes dynamically to align features to classical model specifications.
        if "Config 1" in classical_config:

            user_features = tfidf_abs_uni.transform(
                [user_abstract]
            )

        elif "Config 2" in classical_config:

            user_features = tfidf_abs_bi.transform(
                [user_abstract]
            )

        elif "Config 3" in classical_config:

            # Config 3 was trained on title + abstract features.
            # Since the interface receives only an abstract, the title part is left empty.
            # Matches training shapes by joining an empty sparse title array with the extracted abstract.
            user_title_features = tfidf_ttl.transform(
                [""]
            )

            user_abstract_features = tfidf_abs_bi.transform(
                [user_abstract]
            )

            user_features = sp.hstack(
                [
                    user_title_features,
                    user_abstract_features
                ]
            )

        else:

            return (
                "Doc2Vec interface prediction is not supported in this demo.",
                "",
                "",
                "",
                ""
            )

        # Run classification tasks
        # Handles prediction and mapping tasks if routing choice indicates classification is needed.
        if task_choice in [
            "Classification Only",
            "All Tasks"
        ]:

            # Classical pipeline extraction
            ml_pred_encoded = best_model.predict(
                user_features
            )[0]

            ml_prediction = reverse_mapping[
                ml_pred_encoded
            ]

            # LLM API pipeline routing
            llm_classification_output = classify_structured_prompt(
                user_abstract
            )

            llm_prediction = clean_llm_label(
                llm_classification_output
            )

        # Run summarisation and title-generation tasks
        # Handles generative and extractive summarization loops if requested by user routing.
        if task_choice in [
            "Summarisation Only",
            "All Tasks"
        ]:

            # Traditional Extractive Sentence Retrieval
            classical_summary = extractive_summary(
                user_abstract
            )

            # Abstractive LLM Generation
            llm_generation_output = summarise_and_generate_title(
                user_abstract
            )

            # Parsed structural extraction
            summary = extract_summary(
                llm_generation_output
            )

            title = extract_generated_title(
                llm_generation_output
            )

        return (
            ml_prediction,
            llm_prediction,
            classical_summary,
            summary,
            title
        )

    except Exception as e:

        # Safety fallback boundary catching runtime execution exceptions safely to avoid crashes.
        error_message = f"{type(e).__name__}: {e}"

        return (
            error_message,
            error_message,
            error_message,
            error_message,
            error_message
        )

In [ ]:
# ── 5d: Interface Helper Functions ─────────────────────────────

# Select example abstracts for users to test the interface quickly
# Extracts the first raw abstract for both the Computer Science and Physics categories to seed the UI.
example_cs = (
    df_model[df_model["subject_area"] == "cs"]
    .iloc[0]["abstract"]
)

example_physics = (
    df_model[df_model["subject_area"] == "physics"]
    .iloc[0]["abstract"]
)

# Store examples in a dictionary so dropdown choices can load full abstracts
# Acts as an in-memory cache for mapping simple UI dropdown string labels to raw text payloads.
sample_abstracts = {
    "Computer Science Example": example_cs,
    "Physics Example": example_physics
}


def format_response(outputs):
    """
    Formats model outputs into a readable chatbot response.
    """

    # Unpacks the mixed array of classifications and generated text fields.
    ml_prediction, llm_prediction, classical_summary, llm_summary, title = outputs

    # Assembles a cleanly structured multiline Markdown block for display inside the conversational stream.
    return f"""
Classical ML Predicted Category: {ml_prediction}

LLM Predicted Category: {llm_prediction}

Classical / Extractive Summary:
{classical_summary}

LLM Generated Summary:
{llm_summary}

LLM Generated Title:
{title}
"""


def chat_submit(chat_input, task_choice, history):
    """
    Handles chatbot-style input and appends the interaction to chat history.
    """

    # Coordinates backend analysis execution on the raw text string input by the user.
    outputs = analyse_abstract(
        chat_input,
        task_choice
    )

    # Transforms raw strings and tags into a unified, user-facing narrative output response.
    response = format_response(outputs)

    # Safety catch to ensure state tracking variable is active.
    if history is None:
        history = []

    # Appends the incoming text payload representing the user's conversational turn.
    history.append({
        "role": "user",
        "content": chat_input
    })

    # Appends the formatted pipeline analytical response representing the model's turn.
    history.append({
        "role": "assistant",
        "content": response
    })

    # Returns the individual field strings along with the updated sequential interaction list for Gradio log synchronization.
    return (*outputs, history)


def form_submit(example_choice, task_choice):
    """
    Handles form-style input by running analysis on the selected example abstract.
    """

    # Resolves the actual text block value mapped to the user's UI dropdown key choice.
    selected_abstract = sample_abstracts[
        example_choice
    ]

    # Directly processes the extracted abstract through the active processing framework.
    outputs = analyse_abstract(
        selected_abstract,
        task_choice
    )

    return outputs


def load_form_example(example_choice):
    """
    Loads the selected example abstract into the form textbox.
    """

    # Event hook targeting the text area components to dynamically load selected examples into view.
    return sample_abstracts[
        example_choice
    ]


def switch_assistant_mode(mode):
    """
    Switches visibility between chatbot and form-style assistant layouts.
    """

    # Mutates the runtime visibility settings of competing container blocks to clean up layout presentation.
    if mode == "Chatbot Assistant":

        return (
            gr.update(visible=True),  # Activates conversational chat panel container.
            gr.update(visible=False) # Hides stationary form panel container.
        )

    return (
        gr.update(visible=False), # Hides conversational chat panel container.
        gr.update(visible=True)  # Activates stationary form panel container.
    )

In [ ]:
# ── 5e: Form-style + Conversational Chatbot Assistant ───────────

# Closes any active background Gradio instances to free up ports before launching a new interface.
gr.close_all()


def chatbot_submit(chat_message, history):

    if history is None:
        history = []

    # Normalizes the raw text input for reliable string prefix matching.
    message_lower = chat_message.lower().strip()

    # Maps valid text-based trigger sub-strings to backend pipeline task keys.
    commands = {
        "ml classification:": "Classification Only",
        "llm classification:": "Classification Only",
        "ml summary:": "Summarisation Only",
        "extractive summary:": "Summarisation Only",
        "llm summary:": "Summarisation Only",
        "llm title:": "Summarisation Only",
        "generate title:": "Summarisation Only",
        "all tasks:": "All Tasks"
    }

    matched_command = None

    # Iterates over command mappings to check if the current input prefix matches a pattern.
    for command in commands:

        if message_lower.startswith(command):
            matched_command = command
            break

    # Renders parsing assistance rules directly back to the screen if no valid route trigger is matched.
    if matched_command is None:

        response = (
            "Please start your message with one of these commands:\n\n"
            "- ml classification: <abstract>\n"
            "- llm classification: <abstract>\n"
            "- ml summary: <abstract>\n"
            "- llm summary: <abstract>\n"
            "- llm title: <abstract>\n"
            "- all tasks: <abstract>"
        )

    else:

        # Slices out the pure target text block by removing the parsed command label prefix.
        abstract = chat_message[len(matched_command):].strip()

        # dispatches the isolated document body to the main NLP routing function.
        outputs = analyse_abstract(
            abstract,
            commands[matched_command]
        )

        # Selective conditional extraction: returns only the target text value requested by the user.
        if matched_command == "ml classification:":
            response = f"Classical ML Predicted Category: {outputs[0]}"

        elif matched_command == "llm classification:":
            response = f"LLM Predicted Category: {outputs[1]}"

        elif matched_command in [
            "ml summary:",
            "extractive summary:"
        ]:
            response = f"Classical / Extractive Summary:\n{outputs[2]}"

        elif matched_command == "llm summary:":
            response = f"LLM Generated Summary:\n{outputs[3]}"

        elif matched_command in [
            "llm title:",
            "generate title:"
        ]:
            response = f"LLM Generated Title:\n{outputs[4]}"

        else:
            # Handles 'all tasks:' route by parsing every field into a detailed Markdown string presentation block.
            response = format_response(outputs)

    # Updates conversational logs using standard role-content object definitions for tracking state.
    history.append({
        "role": "user",
        "content": chat_message
    })

    history.append({
        "role": "assistant",
        "content": response
    })

    # Returns an empty string to flush out the user input text area box while synchronizing state.
    return "", history


def switch_interface(assistant_type):

    # Mutates target visibility properties depending on selection choices inside the dropdown manager.
    if assistant_type == "Form-style Assistant":
        return gr.update(visible=True), gr.update(visible=False)

    return gr.update(visible=False), gr.update(visible=True)


# Defines the root wrapper layout component managed by the framework layout engine.
with gr.Blocks(
    title="Scientific Paper NLP Assistant"
) as interface:

    # Global controller selection block allowing toggle control across the presentation layouts.
    assistant_type = gr.Dropdown(
        choices=[
            "Form-style Assistant",
            "Chatbot Assistant"
        ],
        value="Form-style Assistant",
        label="Select Assistant Type"
    )

    # ── Form-style Assistant ───────────────────────────────────

    # Group container representing a traditional, field-oriented static analytical form dashboard.
    with gr.Group(visible=True) as form_section:

        form_interface = gr.Interface(

            fn=analyse_abstract,

            inputs=[
                gr.Textbox(
                    lines=10,
                    label="Enter Scientific Abstract",
                    placeholder="Paste a scientific paper abstract here..."
                ),

                gr.Dropdown(
                    choices=[
                        "All Tasks",
                        "Classification Only",
                        "Summarisation Only"
                    ],
                    value="All Tasks",
                    label="Select NLP Task"
                )
            ],

            outputs=[
                gr.Textbox(
                    label="Classical ML Predicted Category"
                ),

                gr.Textbox(
                    label="LLM Predicted Category"
                ),

                gr.Textbox(
                    label="Classical / Extractive Summary"
                ),

                gr.Textbox(
                    label="LLM Generated Summary"
                ),

                gr.Textbox(
                    label="LLM Generated Title"
                )
            ],

            title="Form-style NLP Assistant",

            description=(
                "Paste or select an abstract from the ArXiv-style dataset "
                "and choose an NLP task."
            ),

            examples=[
                [example_cs, "All Tasks"],
                [example_physics, "All Tasks"]
            ]
        )

    # ── Chatbot Assistant ──────────────────────────────────────

    # Group container wrapping elements into a terminal-style prefix command console interface.
    with gr.Group(visible=False) as chatbot_section:

        gr.Markdown("# Chatbot NLP Assistant")

        gr.Markdown(
            "Paste an abstract from the ArXiv data using one of the commands below. "
            "The chatbot will return only the NLP output you ask for."
        )

        gr.Markdown(
            """
**Commands you can use:**

- `ml classification: <abstract>`
- `llm classification: <abstract>`
- `ml summary: <abstract>`
- `llm summary: <abstract>`
- `llm title: <abstract>`
- `all tasks: <abstract>`
"""
        )

        gr.Markdown("### Sample Abstracts to Copy")

        # Static reference text displays holding benchmark strings ready for clipboard actions.
        gr.Textbox(
            value=example_cs,
            label="Computer Science Sample Abstract",
            lines=5,
            interactive=False
        )

        gr.Textbox(
            value=example_physics,
            label="Physics Sample Abstract",
            lines=5,
            interactive=False
        )

        chat_message = gr.Textbox(
            lines=5,
            label="Chat Input",
            placeholder=(
                "Example: llm summary: paste an abstract from ArXiv data here..."
            )
        )

        with gr.Row():

            chat_submit_btn = gr.Button("Submit")

            chat_clear_btn = gr.Button("Clear")

        # Renders the scrollable operational history block component handling chat logs.
        chatbot = gr.Chatbot(
            label="Chat-style Interaction Window",
            height=450
        )

    # Registers reactive state event hooks linking UI configurations with interactive functions.
    assistant_type.change(
        fn=switch_interface,
        inputs=assistant_type,
        outputs=[
            form_section,
            chatbot_section
        ]
    )

    # Registers actions executing business calculations when selecting interaction triggers.
    chat_submit_btn.click(
        fn=chatbot_submit,
        inputs=[
            chat_message,
            chatbot
        ],
        outputs=[
            chat_message,
            chatbot
        ]
    )

    # Registers lambda reset events to empty out components cleanly upon requests.
    chat_clear_btn.click(
        fn=lambda: ("", []),
        inputs=None,
        outputs=[
            chat_message,
            chatbot
        ]
    )

# Deploys the constructed block definitions to an embedded web server running locally.
interface.launch(
    inline=True,
    debug=False,
    prevent_thread_lock=True
)